In [1]:
import os
from huggingface_hub import snapshot_download

model_name = "Qwen/Qwen2.5-3B-Instruct"

save_dir = os.path.join(
    os.getcwd(),
    "models",
    "Qwen2.5-3B-Instruct"
)

os.makedirs(save_dir, exist_ok=True)

print(f"Model will be downloaded: {save_dir}")

if os.path.exists(save_dir) and os.listdir(save_dir):
    print("Model already exists, skipping download.")
else:
    snapshot_download(
        repo_id=model_name,
        local_dir=save_dir,
        local_dir_use_symlinks=False,
        revision="main"
    )
    print("\nDownloaded successfully!")

print(f"Files: {os.listdir(save_dir)}")

Model will be downloaded: /home/vegunstars/APilus/models/Qwen2.5-3B-Instruct


/home/vegunstars/APilus/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]


Downloaded successfully!
Files: ['README.md', 'config.json', 'LICENSE', 'model-00002-of-00002.safetensors', 'tokenizer_config.json', 'model-00001-of-00002.safetensors', 'generation_config.json', 'merges.txt', 'model.safetensors.index.json', '.gitattributes', 'tokenizer.json', '.cache', 'vocab.json']


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

model_path = os.path.join(
    os.getcwd(),
    "models",
    "Qwen2.5-3B-Instruct"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Model loaded!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded!


In [ ]:
from transformers import GenerationConfig
import time
import torch
from turboquant import TurboQuantCache

def ask(
    prompt,
    max_new_tokens=2048,
    temperature=0.6,
    top_p=0.95,
    top_k=64,
    repetition_penalty=1.1,
    do_sample=True,
    seed=42,
    verbose=True,
):

    if seed is not None:
        torch.manual_seed(seed)

    messages = [
        {"role": "system", "content": "You are a helpful AI coding assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    input_token_count = model_inputs.input_ids.shape[1]

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        top_k=top_k if do_sample else None,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    tq_cache = TurboQuantCache(bits=4)  

    start_time = time.perf_counter()

    generated_ids = model.generate(
        **model_inputs,
        generation_config=gen_config,
        past_key_values=tq_cache,  
        use_cache=True
    )

    elapsed = time.perf_counter() - start_time

    output_ids = generated_ids[0][input_token_count:]

    output_token_count = len(output_ids)

    tokens_per_second = output_token_count / elapsed

    response = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    ).strip()

    if verbose:

        print(f"\n{'─'*48}")
        print(f"🌡 Temperature        : {temperature}")
        print(f"🎯 Top-p             : {top_p}")
        print(f"🔝 Top-k             : {top_k}")
        print(f"🔁 Repetition        : {repetition_penalty}")
        print(f"🎲 Sampling          : {do_sample} | Seed: {seed}")

        print(f"{'─'*42}")

        print(f"📥 Input tokens      : {input_token_count}")
        print(f"📤 Output tokens     : {output_token_count}")

        print(f"⏱ Generation time   : {elapsed:.2f}s")

        print(f"⚡ Speed             : {tokens_per_second:.1f} tok/s")

        print(f"{'─'*48}\n")

    return response


print("✅ ask() ready!")

✅ ask() ready!


In [4]:
print(ask("Who is Acıbadem Universitys Computer Engineering Department Head?", seed=42))

Passing `generation_config` together with generation-related arguments=({'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



────────────────────────────────────────────────
🌡 Temperature        : 0.6
🎯 Top-p             : 0.95
🔝 Top-k             : 50
🔁 Repetition        : 1.1
🎲 Sampling          : True | Seed: 42
──────────────────────────────────────────
📥 Input tokens      : 34
📤 Output tokens     : 109
⏱ Generation time   : 4.14s
⚡ Speed             : 26.3 tok/s
────────────────────────────────────────────────

I don't have real-time access to the latest administrative information or specific details about individual positions at universities, including the Computer Engineering Department head of Acıbadem University. 

Acıbadem University's website and official channels would be the best sources for getting up-to-date information on such roles. For accurate and current details, I recommend visiting their official website or contacting the university directly.

If you need general information about the department itself, I can provide that without needing to check external sources. Would you like me to 

In [5]:
print(ask("Explain TCP vs UDP in detail", seed=42))


────────────────────────────────────────────────
🌡 Temperature        : 0.6
🎯 Top-p             : 0.95
🔝 Top-k             : 50
🔁 Repetition        : 1.1
🎲 Sampling          : True | Seed: 42
──────────────────────────────────────────
📥 Input tokens      : 28
📤 Output tokens     : 512
⏱ Generation time   : 16.45s
⚡ Speed             : 31.1 tok/s
────────────────────────────────────────────────

TCP (Transmission Control Protocol) and UDP (User Datagram Protocol) are both transport layer protocols used for communication over the Internet, but they differ significantly in their approach to handling data transmission.

### 1. **Overview**
- **TCP** is designed for reliable, ordered, and error-checked delivery of data. It ensures that packets arrive at the destination exactly as sent.
- **UDP** is simpler and faster than TCP. It does not provide any reliability or ordering guarantees; it simply sends out datagrams which may get lost, duplicated, or arrive out-of-order.

### 2. **Key Diffe

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

loader = TextLoader("test_rag.json")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'} 
)
vector_db = FAISS.from_documents(chunks, embeddings)

print(f"✅ Vector database built with {len(chunks)} chunks!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector database built with 48 chunks!


In [27]:
from transformers import GenerationConfig
from turboquant import TurboQuantCache
import time
import torch

def ask_with_rag(prompt, max_new_tokens=2048, temperature=0.6, top_p=0.95, top_k=50, repetition_penalty=1.1, do_sample=True, seed=42, verbose=True):
    if seed is not None:
        torch.manual_seed(seed)

    retrieved_docs = vector_db.similarity_search(prompt, k=5)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    
    augmented_prompt = f"Use the following context to answer the question.\n\nContext:\n{context}\n\nQuestion: {prompt}"

    messages = [
        {"role": "system", "content": "You are a helpful University assistant that gives information about University given the relevant pieces of information you'll be trying to answer the question based on the context."},
        {"role": "user", "content": augmented_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_token_count = model_inputs.input_ids.shape[1]

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        top_k=top_k if do_sample else None,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    tq_cache = TurboQuantCache(bits=4)
    start_time = time.perf_counter()

    generated_ids = model.generate(
        **model_inputs,
        generation_config=gen_config,
        past_key_values=tq_cache,
        use_cache=True
    )

    elapsed = time.perf_counter() - start_time
    output_ids = generated_ids[0][input_token_count:]
    output_token_count = len(output_ids)
    tokens_per_second = output_token_count / elapsed

    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    
    if verbose:
        print(f"Input tokens      : {input_token_count} (Includes RAG context)")
        print(f"Speed             : {tokens_per_second:.1f} tok/s")

    return response

In [30]:
print(ask_with_rag("Universitedeki hocalarin isimleri ve unvanlari neler?", seed=42))

Input tokens      : 1932 (Includes RAG context)
Speed             : 25.7 tok/s
Based on the information provided, I can identify some professors and their titles from the text:

1. Prof. Dr. Günaydın Yetik Anacak - He/she received funding for research projects.
2. Dr. Şinasi Can Tadbib Etik Günleri-4 - This appears to be a researcher focused on medical education ethics.
3. Doç. Dr. Özlem İrem Lemmen - She was involved in conducting research at the AP.
4. Dr. Şinasi Can Tadbib Etik Günleri-4 - Another researcher focusing on medical education ethics.

It's important to note that this list is not exhaustive, as there are several other individuals mentioned with their names or initials followed by "etik" or "gunleri", indicating they might also hold positions related to academic ethics or educational practices.

The actual full names and specific roles aren't clearly listed alongside these abbreviations within the provided excerpt. Therefore, without more detailed information, it would be 

In [31]:
print(ask_with_rag("Acibadem universitesi ne zaman kuruldu?", seed=42))

Input tokens      : 1255 (Includes RAG context)
Speed             : 25.9 tok/s
Based on the information provided in the context, Acibadem Üniversitesi was founded in 2007. This can be inferred from the statement mentioning it as an institution that began research projects in 2007. 

However, please note that while this is likely accurate, official university websites or historical records may provide more precise details and could potentially offer different founding dates if there were any delays or changes in the establishment process.
